In [1]:
import numpy as np
import time
import json
import socket
import struct
import select
from pynq import MMIO, allocate, Overlay

In [2]:
ol = Overlay("base.bit")

/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/bitstream.py:133: UserWarning: The provided name 'base.bit' resulted in multiple possible matches:
 - /home/xilinx/jupyter_notebooks/QuickMafs/TestScripts/2k/base.bit
 - /usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/base/base.bit
The first entry of this list, '/home/xilinx/jupyter_notebooks/QuickMafs/TestScripts/2k/base.bit', will be used, please provide the full path in case your target file was a different one in this list.
  warnings.warn(msg, UserWarning)


In [3]:
#base addresses
PIXEL_GEN_BASE  = 0x40000000
DMA_BASE = 0x40400000

#FPGA ----frame ----> AXI-Stream ----write----> DMA
# ┌─────────────────┬────────┬─────────────────────────────────────────────────┐
# │ Name            │ Offset │ Purpose                                         │
# ├─────────────────┼────────┼─────────────────────────────────────────────────┤
# │ S2MM_DMACR      │  0x30  │ Control Register — tells the DMA how to behave  │
# │ S2MM_DMASR      │  0x34  │ Status Register  — reflects DMA state / errors  │
# │ S2MM_DA         │  0x48  │ Destination Address — where in RAM to write data│
# │ S2MM_LENGTH     │  0x58  │ Transfer length (bytes) — writing this STARTS   │
# │                 │        │ the transfer; DMA then waits for stream data    │
# └─────────────────┴────────┴─────────────────────────────────────────────────┘
#
# ── S2MM_DMACR (0x30) — Control Register ──────────────────────────────────────
#   bit  0   RS          Run/Stop.  1 = channel running, 0 = channel stopped/halted.
#                        Must be set to 1 before writing S2MM_LENGTH.
#   bit  2   Reset       Soft-reset the S2MM channel.  Write 1; hardware self-clears
#                        within a few microseconds.  Clears all state from prior transfers.
#   bit 12   IOC_IrqEn   Enable "interrupt on completion" (IOC).  When the transfer
#                        finishes, IOC_Irq (DMASR bit 12) is set.
#   [23:16]  IRQThreshold  Number of completions before the interrupt fires.
#                        Set to 1 so every single transfer raises an interrupt.
#
# ── S2MM_DMASR (0x34) — Status Register ───────────────────────────────────────
#   bit  0   Halted      1 when the channel is stopped (RS=0 or after an error).
#                        Should go to 0 shortly after writing RS=1.
#   bit  1   Idle        1 when no transfer is in progress.
#   bit  4   DMAIntErr   Internal error.  Most common cause: the byte count in
#                        S2MM_LENGTH doesn't match where the FPGA asserts TLAST
#                        on the AXI-Stream.  Check the byte count matches FPGA output.
#   bit  6   SGDecErr    Scatter-Gather descriptor decode error.  Only relevant in
#                        SG mode — if this fires in Simple mode, the DMA is
#                        misconfigured (set to SG in Vivado when it should be Simple).
#   bit  7   SGSlvErr    Scatter-Gather slave error (similar: SG mode only).
#   bit 12   IOC_Irq     Interrupt-on-Completion flag.  Hardware sets this when the
#                        transfer finishes normally.  Write-1-to-Clear (W1C).
#   bit 14   Err_Irq     Summary error interrupt.  Set whenever any error bit fires.
#                        W1C.
#
# ── S2MM_DA (0x48) — Destination Address ──────────────────────────────────────
#   The physical (device) RAM address the DMA writes incoming stream bytes into.
#   Must be a physically contiguous buffer — use pynq.allocate(), NOT a regular
#   numpy array (which can be fragmented across pages and unusable by the DMA).
#   Obtain the address with buf.device_address after allocation.
#
# ── S2MM_LENGTH (0x58) — Transfer Length ──────────────────────────────────────
#   How many bytes to receive from the AXI-Stream.  Writing this register is the
#   "go" signal — the DMA immediately arms itself and waits for stream data.
#   For correctness, always write DMACR and DA *before* writing LENGTH.

S2MM_DMACR  = 0x30
S2MM_DMASR  = 0x34
S2MM_DA     = 0x48
S2MM_LENGTH = 0x58

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# FRAME DIMENSIONS
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
#Height must be a multiple of sixty as it is = BAND_H,
#and the largest supported frame is 2048 x 1080.

DEFAULT_W, DEFAULT_H = 160, 120
NUM_LANES = 12
#band streaming params
BAND_ROWS = 5
BAND_H = NUM_LANES * BAND_ROWS
MAX_W, MAX_H = 2048, 1080

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# Memory Mapped Input/Output
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# MMIO init of pixel_gen and dma
pg  = MMIO(PIXEL_GEN_BASE,  0x1000)
dma = MMIO(DMA_BASE, 0x10000)

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# FIXED-POINT HELPERS
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
#converts input to 18 bit q4.14 implementation
def q(x):
    #mul by 2^14 and pass it through bitmap
    return int(np.round(x * 16384)) & 0x3FFFF

def pgw(reg, val):
    #writes value to register. each reg is 32 bits wide so offset is reg id * 4
    pg.write(reg * 4, int(val))

def pgr(reg):
    #read from regf
    return pg.read(reg * 4)

def set_resolution(w, h):
    #write resolution to fpga
    #band streaming: reg33 = band_pix (rows one lane buffers x width), reg34 = nbands
    band_pix = BAND_ROWS * w
    nbands   = h // BAND_H
    pgw(31, w)
    pgw(32, h)
    pgw(33, band_pix)
    pgw(34, nbands)

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHYSICS PARAMETER WRITER
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# AXI-Lite register map for pixel_generator_0:
#  Reg  0        control        start  (1 = start, 0 = hold)
#  Reg  1        x_min          x_min
#  Reg  2        y_min          y_min
#  Reg  3        dx             width per pixel x_range / w
#  Reg  4        dy             width per pixel y_range / h
#  Reg  5, 6     mag0 x, y      magnet 0 x, y
#  Reg  7, 8     mag1 x, y      magnet 1 x, y
#  Reg  9, 10    mag2 x, y      magnet 2 x, y
#  Reg 11        gamma          velocity damping coefficient
#  Reg 12        omega2         omega squared
#  Reg 13        h2             height squared
#  Reg 14        mu             magnetic strength coefficient
#  Reg 15        dt             simulation time step
#  Reg 16        r_settle^2     pendulum settling distance limit
#  Reg 17        v_settle       pendulum settling velocity limit
#  Reg 18        r_settle^2+h^2 precomputed term for nearest-magnet check
#  Reg 19        misc           [31:12] = n_magnets (3), [11:0] = max_steps (4000)
#  Reg 30        mag_active     [2:0] = active-magnet mask (bit i = magnet i; 000 => all active)
#  Reg 31-34     resolution     img_w / img_h / band_pix(pixels per band) / nbands

def write_physics_params(
    magnets,
    x_min,
    y_min,
    x_range,
    y_range,
    w,
    h,
    gamma,
    omega2,
    h2,
    mu,
    dt=0.01,
    r_settle=0.25,
    active=(1, 1, 1),
):
    #all floats are translated to q4.14
    #set to hold
    pgw(0, 0)

    pgw(1, q(x_min))
    pgw(2, q(y_min))
    pgw(3, q(x_range / w)) #dx
    pgw(4, q(y_range / h)) #dy

    if isinstance(magnets, dict):
        mag_list = list(magnets.values())
    else:
        mag_list = list(magnets)

    default_mags = [
        {'x':  1.0, 'y':  0.0  },
        {'x': -0.5, 'y':  0.866},
        {'x': -0.5, 'y': -0.866},
    ]
    while len(mag_list) < 3:
        mag_list.append(default_mags[len(mag_list)])

    pgw(5,  q(mag_list[0]['x']))
    pgw(6,  q(mag_list[0]['y']))
    pgw(7,  q(mag_list[1]['x']))
    pgw(8,  q(mag_list[1]['y']))
    pgw(9,  q(mag_list[2]['x']))
    pgw(10, q(mag_list[2]['y']))

    pgw(11, q(gamma))
    pgw(12, q(omega2))
    pgw(13, q(h2))
    pgw(14, q(mu))
    pgw(15, q(dt))

    pgw(16, q(r_settle ** 2))
    pgw(17, q(0.05))
    pgw(18, (q(r_settle ** 2) + q(h2)) >> 1)
    pgw(19, (3 << 12) | 4000) # [31:12] = 3 magnets,  [11:0] = 4000 max steps

    #write active magnet mask
    mask = sum((1 << i) for i in range(3) if active[i])
    pgw(30, mask)


# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# DMA ARM
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
def arm_dma(phys_addr, n_bytes):
    #prepare the DMA channel to receive one frame (n_bytes) from the FPGA
    
    #reset DMA DMACONTROL[2] = reset
    dma.write(S2MM_DMACR, 0x00000004)   # DMACR[2] = Reset

    #Poll until hardware clears reset bit
    t0 = time.time()
    while dma.read(S2MM_DMACR) & 0x4:
        if time.time() - t0 > 1.0:
            print("DMA RESET WAS NOT CLEARED")
            break
        time.sleep(0.001)

    #clears all error bits
    #writing 0x7000 clears all three error bits so we start with a clean status register.
    dma.write(S2MM_DMASR, 0x00007000)


    #channel config
    # 0x01001001 breakdown:
    # bit  0 = 1 -> Run/Stop = 1 -> start running channel
    # bit 12 = 1  -> IOC_IrqEn = 1 -> raise interrupt when complete
    # bits [23:16] = 1 ->  IRQThreshold =1 -> interrupt after 1 completion
    dma.write(S2MM_DMACR, 0x01001001)

    #set destination address to the buffer address
    dma.write(S2MM_DA, phys_addr)

    #write byte count -- this is the "go" signal
    dma.write(S2MM_LENGTH, n_bytes)


# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# DMA POLL
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
def poll_dma(timeout=300.0):
    #polls for DMA channel to finish
    #returns tuple (done, elasped_seconds)
    #done=True  → IOC_Irq fired
    #done=False → timeout/error
    
    #read init
    
    t0      = time.time()
    sr_init = dma.read(S2MM_DMASR)

    while time.time() - t0 < timeout:
        sr  = dma.read(S2MM_DMASR)
        fdl = pgr(20) & 0x1   # pixel_generator frame_done_latch: 1 = FPGA finished its frame
        
        #success
        if (sr & 0x1000) and not (sr_init & 0x1000):
            return True, time.time() - t0

        #error code
        if sr & 0x4010:
            print("DMA ERROR")
            return False, time.time() - t0

        time.sleep(0.001) #1 ms poll interval keeps low-res renders responsive
        
    return False, timeout #timed out


# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PIXEL BUFFER DECODING
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
#DECODE MAP:
#  bits [1:0]  magnet_id  0 means timedout
#  bits [5:2]  step_cat   convergence categroy (0 = timed out, higher the number, faster the convergence)

def decode_buffer(raw, n_pixels):
    pixels   = np.array(raw[:n_pixels], dtype=np.uint8)
    mag_id   = (pixels & 0x3).astype(np.uint8) #lower 2 bits = magent id 0x3 = 11
    step_cat = ((pixels >> 2) & 0xF).astype(np.uint8) #the other 4 bits  = step_cateogry

    #remap if needed:
    remap  = np.array([0, 1, 2, 3], dtype=np.uint8)
    mag_id = remap[mag_id]

    return pixels, mag_id, step_cat

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TCP server that listens for connections from Unity, receives physics params,
# magnet positions, and viewport info, triggers FPGA rendering, and sends rendered
# frames back to unity
#
# board is the server, unity vonnects as a client.
# ══════════════════════════════════════════════════════════════════════════════

# host, port definitions
HOST, PORT = "0.0.0.0", 12345

# Message type ID for protocol encrypted in message header - defines what type payload is
# 0x01: Unity -> PYNQ : Physics params JSON {damping_factor, magnetic_strength, pendulum_length, pendulum_height, resX, resY}
# 0x02: Unity -> PYNQ : Magnet positions JSON {magnets: {magnet_i: {x, y}}}
# 0x03: Unity -> PYNQ : Viewport JSON {x_min, x_max, y_min, y_max}
# 0x04: Unity -> PYNQ : ID of pixel for trajectory visualization
# 0x10: PYNQ -> Unity : u16 w, u16 h, u8 bitDepth, u32 version, pixels
# 0x14: PYNQ -> Unity : u32 pixel_id, u16 n, then n×(f32 x, f32 y)
MSG_PARAMS, MSG_MAGNETS, MSG_VIEWPORT, MSG_IMAGE = 0x01, 0x02, 0x03, 0x10
MSG_TRAJ_REQ, MSG_TRAJ = 0x04, 0x14 
BIT_DEPTH = 6 # 2 bits for magnet ID, 4 bits for step category

# default ranges for viewport
X_MIN, Y_MIN = -1.8, -1.8
X_RANGE = Y_RANGE = 3.6

# default values before recieving values from unity
DEFAULT_PARAMS = {'mu': 1.0, 'gamma': 0.2, 'h2': 0.25, 'omega2': 1.0}
DEFAULT_MAGNETS = [
    {'x':  1.0, 'y':  0.0  },
    {'x': -0.5, 'y':  0.866},
    {'x': -0.5, 'y': -0.866},
]

#fss epsilon value specified by david
EPSILON_FSS = 0.1


# =============================================================================================
# HELPER FUNCTIONS FOR TCP
# =============================================================================================

# read exactly n bytes from the conn socket, handling partial reads and disconnects
def recv_exactly(conn, n):
    buf = bytearray()
    while len(buf) < n:
        chunk = conn.recv(n - len(buf))
        #connection check
        if not chunk:
            raise ConnectionError("Unity disconnected")
        buf.extend(chunk)
    return bytes(buf)

#reads message frame (type and payload) from the connection
def read_frame(conn):
    # read 5 byte header: 1 byte type, 4 byte length
    #big endian. msg_type is u8, length is u32
    msg_type, length = struct.unpack(">BI", recv_exactly(conn, 5))
    payload = recv_exactly(conn, length) if length else b""
    return msg_type, payload

# sends a message frame (big endian unsigned int for type and length) + payload to connection
def send_frame(conn, msg_type, payload):
    conn.sendall(struct.pack(">BI", msg_type, len(payload)) + payload)

def socket_has_data(conn, timeout=0.0):
      return bool(select.select([conn], [], [], timeout)[0])
def apply_unity_message(msg_type, payload, params, magnets, active_magnets, viewport, resolution):
    body = json.loads(payload.decode("utf-8")) if payload else {}
    request_id = int(body.get("requestId", 0))
    if msg_type == MSG_PARAMS:
        params = params_from_unity(body)
        resolution = resolution_from_unity(body, resolution)
        return params, magnets, active_magnets, viewport, resolution, request_id, True
    if msg_type == MSG_MAGNETS:
        magnets = magnets_from_unity(body)
        active_magnets = active_from_unity(body)
        return params, magnets, active_magnets, viewport, resolution, request_id, True
    if msg_type == MSG_VIEWPORT:
        viewport = viewport_from_unity(body)
        return params, magnets, active_magnets, viewport, resolution, request_id, True
    return params, magnets, active_magnets, viewport, resolution, request_id, False
def process_one_message(conn, params, magnets, active_magnets, viewport, resolution, blocking=True):
    if not blocking and not socket_has_data(conn):
        return params, magnets, active_magnets, viewport, resolution, 0, False, False
    msg_type, payload = read_frame(conn)
    body = json.loads(payload.decode("utf-8")) if payload else {}
    if msg_type == MSG_TRAJ_REQ:
        w, h = resolution
        handle_traj_request(conn, int(body["pixel_id"]), params, magnets, viewport, w, h, tuple(active_magnets))
        return params, magnets, active_magnets, viewport, resolution, 0, False, True
    params, magnets, active_magnets, viewport, resolution, request_id, render = apply_unity_message(
    msg_type, payload, params, magnets, active_magnets, viewport, resolution)
    return params, magnets, active_magnets, viewport, resolution, request_id, render, False


#unity payload decoding
def params_from_unity(body):
    L = float(body["pendulumLength"])
    return {
        'mu':     float(body["magneticStrength"]),  # reg 14
        'gamma':  float(body["dampingFactor"]),     # reg 11
        'h2':     float(body["pendulumHeight"]) ** 2,  # reg 13
        'omega2': 9.81 / L if L > 0 else 9.81,      # reg 12
        'fss':    bool(body.get("fss", False)),     # final state sensitivity mode
    }

#resolution decode from unity payload
def resolution_from_unity(body, current):
    w, h = current
    return (int(body.get("resX", w)), int(body.get("resY", h)))

#magnet position deocde from unity payload
def magnets_from_unity(body):
    # body = {"magnets": {"magnet_0": {"x": mag0x,"y":mag0y}, {"magnet_1": {"x": mag1x,"y":mag1y}, .....}}
    mags = body.get("magnets", {})
    ordered = sorted(mags.items(), key=lambda kv: kv[0])  #sort by magnet id through lambda
    return [{'x': float(m['x']), 'y': float(m['y'])} for _, m in ordered]

#active magnets = whichever magnet_i keys unity actually sent (default all on)
def active_from_unity(body):
    mags = body.get("magnets", {})
    if not mags:
        return [1, 1, 1]
    active = [0, 0, 0]
    for key in mags:
        idx = int(str(key).split("_")[-1])  #pop last index: "magnet_2" -> 2
        if 0 <= idx < 3:
            active[idx] = 1
    return active

#decode pan/zoom (viewport) from unity payload
def viewport_from_unity(body):
    #get keys from body with fallback to defaults
    x_min = float(body.get('x_min', X_MIN))
    x_max = float(body.get('x_max', x_min + X_RANGE))
    y_min = float(body.get('y_min', Y_MIN))
    y_max = float(body.get('y_max', y_min + Y_RANGE))
    return {
        'x_min': x_min,
        'y_min': y_min,
        'x_range': x_max - x_min,
        'y_range': y_max - y_min,
    }



#run one frame on the fpga and decode it -> (mag_id, cat, done, elapsed)
def _run_basin(p, magnets, viewport, buf, w, h, active=(1, 1, 1)):
    n_pixels = w * h
    set_resolution(w, h)
    write_physics_params(
        magnets = magnets if magnets else DEFAULT_MAGNETS,
        x_min   = viewport['x_min'],
        y_min   = viewport['y_min'],
        x_range = viewport['x_range'],
        y_range = viewport['y_range'],
        w       = w,
        h       = h,
        gamma   = p['gamma'],
        omega2  = p['omega2'],
        h2      = p['h2'],
        mu      = p['mu'],
        active  = active,
    )
    arm_dma(buf.device_address, n_pixels)
    pgw(0, 1) # start FPGA
    done, elapsed = poll_dma()
    pgw(0, 0) #safety fpga stop
    buf.invalidate() #invalidates cpu cache of buffer

    raw = np.array(buf, dtype=np.uint8)
    #decode pixel_id, magnet_id and category from raw buffer
    pixels, mag_id, cat = decode_buffer(raw, n_pixels)
    return mag_id.astype(np.int16), cat.astype(np.uint8), done, elapsed


#render one frame at the given (w, h) and return the bytes to send back to unity
def render_frame(p, magnets, viewport, buf, w, h, active=(1, 1, 1)):
    #normal basin render: bits[5:2]=step_cat, bits[1:0]=magnet_id
    #if fss is off, just run one basin/frame
    if not p.get('fss', False):
        mag_id, cat, done, elapsed = _run_basin(p, magnets, viewport, buf, w, h, active)
        out = (((cat & 0xF) << 2) | (mag_id & 0x3)).astype(np.uint8)
        return out.tobytes(), done, elapsed #also send done and elapsed for debugging

    #code from here is fss from david
    eps = EPSILON_FSS
    vp_plus  = {**viewport, 'x_min': viewport['x_min'] + eps}
    vp_minus = {**viewport, 'x_min': viewport['x_min'] - eps}

    #id, category, done, elapsed for each of the three frames
    base_id,  base_cat,  d0, e0 = _run_basin(p, magnets, viewport, buf, w, h, active)
    plus_id,  plus_cat,  d1, e1 = _run_basin(p, magnets, vp_plus,  buf, w, h, active)
    minus_id, minus_cat, d2, e2 = _run_basin(p, magnets, vp_minus, buf, w, h, active)

    #check timeouts and sensitivity
    timeout_any = (base_cat == 0) | (plus_cat == 0) | (minus_cat == 0)
    sensitive   = (~timeout_any) & ((base_id != plus_id) | (base_id != minus_id))

    #FSS word: bits[5:2]=base step cat, bit[1]=timeout_any, bit[0]=sensitive
    out = (((base_cat & 0xF) << 2) | (timeout_any.astype(np.uint8) << 1) | sensitive.astype(np.uint8)).astype(np.uint8)
    done    = d0 and d1 and d2
    elapsed = e0 + e1 + e2
    return out.tobytes(), done, elapsed

# =============================================================================================
# HELPER FUNCTIONS FOR TRAJECTORY
# =============================================================================================
def to_signed32(v):
    return v - (1 << 32) if (v & (1 << 31)) else v

def read_trajectory():
    n = min(int(pgr(25)), 4096)  # trajectory length, saturated at 4096
    xs = np.empty(n, np.float32); ys = np.empty(n, np.float32)
    for step in range(n):
        pgw(22, step)  # read address
        xs[step] = to_signed32(pgr(23)) / 16384.0   # x coordinate
        ys[step] = to_signed32(pgr(24)) / 16384.0   # y coordinate
    return xs, ys

def poll_traj_done(timeout=30.0):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if pgr(26) & 0x1: # traj_done (target pixel settled/timed out)
            return True
        time.sleep(0.001)
    return False

def send_trajectory(conn, pixel_id, xs, ys):
    n = len(xs)
    payload = struct.pack(">IH", int(pixel_id), n) # 32 bits for pixel ID, 16 bits for n
    pts = np.empty(2 * n, np.float32) # twice the size because stores both x and y coordinates
    pts[0::2] = xs; pts[1::2] = ys # even locations contain x, odd locations contain y
    payload += pts.astype(">f4").tobytes() # n × (f32 x, f32 y), big-endian
    send_frame(conn, MSG_TRAJ, payload)

def handle_traj_request(conn, pid, p, magnets, viewport, w, h, active=(1,1,1)):
    set_resolution(w, h)
    pgw(21, int(pid)) # traj_px_id
    write_physics_params(
        magnets=magnets if magnets else DEFAULT_MAGNETS,
        x_min=viewport['x_min'], y_min=viewport['y_min'],
        x_range=viewport['x_range'], y_range=viewport['y_range'],
        w=w, h=h,
        gamma=p['gamma'], omega2=p['omega2'], h2=p['h2'], mu=p['mu'],
        active=active,
    )

    #arm dma first
    arm_dma(buf1.device_address, w * h)
    pgw(0, 1) # start computation
    poll_traj_done(timeout=300.0) # wait for pixel to reach settled/timeout
    xs, ys = read_trajectory()
    pgw(0, 0) # end computation
    send_trajectory(conn, pid, xs, ys)

#create tcp socket
srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
#allow for quick restarts should connection fail
srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

#bind to host and port, listen for connections
srv.bind((HOST, PORT))
#accept ONE connection - unity will be only client
srv.listen(1)
print(f"Listening on {HOST}:{PORT}")

#allocate largest possible resolution for buffer
buf1 = allocate(shape=(MAX_W * MAX_H,), dtype=np.uint8)

#frame version for unity - debug
version = 0

try:
    while True:
        #block and wait for unity to connect, then set TCP_NODELAY for low latency - send frames immediantely
        conn, addr = srv.accept()
        conn.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)

        print("Unity connected at adddress:", addr)

        #reset params to prevent stale values
        params   = dict(DEFAULT_PARAMS)
        magnets  = [dict(m) for m in DEFAULT_MAGNETS]
        active_magnets = [1, 1, 1]
        viewport = {'x_min': X_MIN, 'y_min': Y_MIN, 'x_range': X_RANGE, 'y_range': Y_RANGE}
        resolution = (DEFAULT_W, DEFAULT_H)
        last_rendered = None
        try:
            while True:
                params, magnets, active_magnets, viewport, resolution, request_id, should_render, _ = process_one_message(
                  conn, params, magnets, active_magnets, viewport, resolution, blocking=True)
                if not should_render:
                      continue
                # Drain queued messages and wait briefly for back-to-back updates
                latest_request_id = request_id
                while socket_has_data(conn, 0.002):
                    params, magnets, active_magnets, viewport, resolution, request_id, _, _ = process_one_message(
                      conn, params, magnets, active_magnets, viewport, resolution, blocking=False)
                    if request_id > latest_request_id:
                          latest_request_id = request_id
                state = (dict(params), [dict(m) for m in magnets], dict(viewport), tuple(active_magnets), resolution)
                if state == last_rendered:
                      continue
                w, h = resolution
                pixels, done, elapsed = render_frame(params, magnets, viewport, buf1, w, h, active_magnets)
                # If Unity sent newer data while we were rendering, discard this frame
                if socket_has_data(conn):
                    print(f"  skip stale frame req={latest_request_id} (newer messages queued)")
                    continue
                last_rendered = state
                version += 1
                img = struct.pack(">HHBI", w, h, BIT_DEPTH, version) + pixels
                send_frame(conn, MSG_IMAGE, img)
                print(f"  frame v{version} req={latest_request_id} {w}x{h} done={done} ({elapsed:.3f}s)")

        except (ConnectionError, OSError) as e:
            print("  connection closed:", e)
        finally:
            conn.close()
            #wait for next connection

except KeyboardInterrupt:
    print("Server stopped.")
finally:
    srv.close()
    #kill server and close socket

Listening on 0.0.0.0:12345
Unity connected at adddress: ('192.168.2.1', 61660)
  frame v1 req=1 120x120 done=True (0.226s)
  frame v2 req=2 120x120 done=True (0.213s)
  frame v3 req=3 120x120 done=True (0.214s)
  frame v4 req=4 120x120 done=True (0.202s)
  frame v5 req=5 120x120 done=True (0.219s)
  frame v6 req=6 120x120 done=True (0.236s)
  frame v7 req=7 120x120 done=True (0.134s)
  frame v8 req=8 120x120 done=True (0.104s)
  frame v9 req=9 120x120 done=True (0.091s)
  frame v10 req=10 48x120 done=True (0.042s)
  frame v11 req=11 120x288 done=True (0.182s)
  frame v12 req=12 120x120 done=True (0.091s)
  frame v13 req=13 276x288 done=True (0.406s)
  frame v14 req=14 120x120 done=True (0.092s)
  frame v15 req=15 384x384 done=True (0.824s)
  connection closed: Unity disconnected
Unity connected at adddress: ('192.168.2.1', 62724)
  frame v16 req=1 120x120 done=True (0.226s)
  skip stale frame req=3 (newer messages queued)
  skip stale frame req=7 (newer messages queued)
  skip stale fr